## Config paths/models

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import json

PROJECT_ROOT = Path("/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/")
DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")

DATA_PROCESSED = DATA_ROOT / "processed"
RESULTS_DIR = DATA_ROOT / "results"
MODELS_DIR = DATA_ROOT / "models"
CHECKPOINTS_DIR = MODELS_DIR / "checkpoints"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"

checkpoint_id = (
    "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
    "_CNN_Pool"
    "_pool4_emb64_mse"
    "_MSELoss"
    "_seed123"
)

pred_path = RESULTS_DIR / f"{checkpoint_id}_train_val_predictions_embeddings.npz"
history_path = RESULTS_DIR / f"{checkpoint_id}_history.npz"
checkpoint_path = CHECKPOINTS_DIR / f"{checkpoint_id}_checkpoint.pt"

assert pred_path.exists(), pred_path

print("pred_path:", pred_path)
print("history_path exists:", history_path.exists())
print("checkpoint_path exists:", checkpoint_path.exists())

PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
PROJECT_ROOT exists: True
hostname: pcae159.ciemat.es
python: /afs/ciemat.es/user/v/vserrano/miniconda3/envs/gw-env/bin/python
cwd: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe
DATA_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/data
sys.path[:3]: ['/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe', '/afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks', '/opt/utils']


## Load the preds(/embs)

In [ ]:
data = np.load(pred_path, allow_pickle=True)

print(data.files)

pred_train = data["pred_train"]
y_train = data["y_train"]
emb_train = data["emb_train"]

pred_val = data["pred_val"]
y_val = data["y_val"]
emb_val = data["emb_val"]

y_mean = data["y_mean"]
y_std = data["y_std"]

train_idx = data["train_idx"]
val_idx = data["val_idx"]

label_names = data["label_names"].tolist()

dataset_path = Path(str(data["dataset_path"]))
split_path = Path(str(data["split_path"]))
label_stats_path = Path(str(data["label_stats_path"]))
checkpoint_file = str(data["checkpoint_file"])

print("pred_train:", pred_train.shape)
print("y_train:", y_train.shape)
print("emb_train:", emb_train.shape)

print("pred_val:", pred_val.shape)
print("y_val:", y_val.shape)
print("emb_val:", emb_val.shape)

print("y_mean:", y_mean)
print("y_std:", y_std)
print("label_names:", label_names)

print("dataset_path:", dataset_path)
print("checkpoint_file:", checkpoint_file)

## (Standarized) Metrics

In [ ]:
from src.models.evaluate import regression_metrics, inverse_standardize

metrics_train_std = regression_metrics(
    y_true=y_train,
    y_pred=pred_train,
    label_names=label_names,
    split_name="train_std",
)

metrics_val_std = regression_metrics(
    y_true=y_val,
    y_pred=pred_val,
    label_names=label_names,
    split_name="val_std",
)

metrics_all_std = pd.concat(
    [metrics_train_std, metrics_val_std],
    ignore_index=True,
)

metrics_all_std

In [ ]:
print("train global MSE:", metrics_train_std["MSE"].mean())
print("val global MSE:", metrics_val_std["MSE"].mean())

## (Physical) Metrics

In [ ]:
pred_train_phys = inverse_standardize(pred_train, y_mean, y_std)
y_train_phys = inverse_standardize(y_train, y_mean, y_std)

pred_val_phys = inverse_standardize(pred_val, y_mean, y_std)
y_val_phys = inverse_standardize(y_val, y_mean, y_std)

metrics_train_phys = regression_metrics(
    y_true=y_train_phys,
    y_pred=pred_train_phys,
    label_names=label_names,
    split_name="train_phys",
)

metrics_val_phys = regression_metrics(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys",
)

metrics_all_phys = pd.concat(
    [metrics_train_phys, metrics_val_phys],
    ignore_index=True,
)

metrics_all_phys

## Training curve

In [ ]:
if history_path.exists():
    history = np.load(history_path, allow_pickle=True)

    print(history.files)

    train_loss = history["train_loss"]
    val_loss = history["val_loss"]

    plt.figure(figsize=(7, 4))
    plt.plot(train_loss, label="train")
    plt.plot(val_loss, label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    print("best val loss from history:", np.min(val_loss))
else:
    print("No history file found:", history_path)

## Evaluation Plots

In [ ]:
from src.models.plots import plot_pred_vs_true, plot_residuals, plot_residual_vs_true, plot_abs_error_vs_quantity

### Pred vs True

In [ ]:
plot_pred_vs_true(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys",
)


### Residuals

In [ ]:
plot_residuals(y_val_phys, pred_val_phys, label_names, "val_phys")
plot_residual_vs_true(y_val_phys, pred_val_phys, label_names, "val_phys")

### Error vs SNR

In [ ]:
with h5py.File(dataset_path, "r") as f:
    print(list(f.keys()))
    print(list(f["snr"].keys()))

    snr_all = f["snr/network"][:]
    snr_train = snr_all[train_idx]
    snr_val = snr_all[val_idx]

print("snr_train:", snr_train.min(), snr_train.max())
print("snr_val:", snr_val.min(), snr_val.max())

In [ ]:
plot_abs_error_vs_quantity(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="val_phys",
)

### Error vs parameters (mass, distance, ...)

In [ ]:
with h5py.File(dataset_path, "r") as f:
    mass_1_all = f["parameters/mass_1"][:]
    mass_2_all = f["parameters/mass_2"][:]
    distance_all = f["parameters/distance"][:]

mass_1_val = mass_1_all[val_idx]
mass_2_val = mass_2_all[val_idx]
distance_val = distance_all[val_idx]

In [ ]:
plot_abs_error_vs_quantity(
    quantity=distance_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="distance",
    split_name="val_phys",
)

For all labels:

In [ ]:
for j, label in enumerate(label_names):
    plot_abs_error_vs_quantity(
        quantity=y_val_phys[:, j],
        y_true=y_val_phys,
        y_pred=pred_val_phys,
        label_names=label_names,
        quantity_name=f"true {label}",
        split_name="val_phys",
    )

## Save the metrics (cvs)

In [ ]:
eval_dir = RESULTS_DIR / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

metrics_std_path = eval_dir / f"{checkpoint_id}_metrics_std.csv"
metrics_phys_path = eval_dir / f"{checkpoint_id}_metrics_phys.csv"

metrics_all_std.to_csv(metrics_std_path, index=False)
metrics_all_phys.to_csv(metrics_phys_path, index=False)

print("Saved:", metrics_std_path)
print("Saved:", metrics_phys_path)

## Comparing models

In [ ]:
prediction_files = sorted(RESULTS_DIR.glob("*train_val_predictions_embeddings.npz"))

for p in prediction_files:
    print(p.name)

In [ ]:
from src.models.utils import summarize_prediction_file

summary_rows = [summarize_prediction_file(p) for p in prediction_files]

summary_df = pd.DataFrame(summary_rows).sort_values("val_MSE_global")
summary_df


## 2. Plot pred vs true

Qué mirar:

- Si los puntos siguen la diagonal.
- Si hay saturación en masas altas.
- Si hay regresión a la media.
- Si chi_eff está comprimido cerca de cero.

Mi predicción: chi_eff tendrá bastante regresión a la media.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_pred_vs_true(y_true, y_pred, label_names, split_name="test"):
    for j, label in enumerate(label_names):
        true = y_true[:, j]
        pred = y_pred[:, j]

        min_val = min(true.min(), pred.min())
        max_val = max(true.max(), pred.max())

        plt.figure(figsize=(8, 8))
        plt.scatter(true, pred, s=12, alpha=0.6)
        plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Predicted {label}")
        plt.title(f"{split_name}: predicted vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_pred_vs_true(
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    split_name="val_phys"
)

## 3. Residuals per label

In [ ]:
def plot_residuals(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.hist(residual[:, j], bins=40, alpha=0.8)
        plt.axvline(0.0, linestyle="--")
        plt.xlabel(f"Residual: pred - true ({label})")
        plt.ylabel("Count")
        plt.title(f"{split_name}: residual distribution — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_residuals(y_val_phys, pred_val_phys, label_names, "val_phys")

In [ ]:
def plot_residual_vs_true(y_true, y_pred, label_names, split_name="test"):
    residual = y_pred - y_true

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], residual[:, j], s=12, alpha=0.6)
        plt.axhline(0.0, linestyle="--")
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Residual: pred - true")
        plt.title(f"{split_name}: residual vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

Esto es más importante que el histograma. Te dirá dónde falla el modelo.

In [ ]:
plot_residual_vs_true(y_val_phys, pred_val_phys, label_names, "val_phys")

## 4. Absolute error vs true value

Este plot es clave para Mondrian. Si el error aumenta con masa o depende de chi_eff, entonces tiene sentido usar bins condicionados.

In [ ]:
def plot_abs_error_vs_true(y_true, y_pred, label_names, split_name="test"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(y_true[:, j], abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(f"True {label}")
        plt.ylabel(f"Absolute error")
        plt.title(f"{split_name}: absolute error vs true — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_true(y_val_phys, pred_val_phys, label_names, "val_phys")

## 5. Error vs SNR

In [ ]:
snr_val = network_snrs[val_idx]

In [ ]:
def plot_abs_error_vs_quantity(quantity, y_true, y_pred, label_names, quantity_name, split_name="val"):
    abs_error = np.abs(y_pred - y_true)

    for j, label in enumerate(label_names):
        plt.figure(figsize=(6, 4))
        plt.scatter(quantity, abs_error[:, j], s=12, alpha=0.6)
        plt.xlabel(quantity_name)
        plt.ylabel(f"Absolute error in {label}")
        plt.title(f"{split_name}: abs error vs {quantity_name} — {label}")
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
plot_abs_error_vs_quantity(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    quantity_name="network SNR",
    split_name="val_phys"
)

For bins

In [ ]:
def metrics_by_quantity_bins(quantity, y_true, y_pred, label_names, bin_edges, quantity_name):
    rows = []

    for b in range(len(bin_edges) - 1):
        lo = bin_edges[b]
        hi = bin_edges[b + 1]

        mask = (quantity >= lo) & (quantity < hi)

        if mask.sum() == 0:
            continue

        residual = y_pred[mask] - y_true[mask]
        abs_error = np.abs(residual)

        for j, label in enumerate(label_names):
            rows.append({
                "quantity": quantity_name,
                "bin": f"[{lo:.2f}, {hi:.2f})",
                "count": int(mask.sum()),
                "label": label,
                "MAE": abs_error[:, j].mean(),
                "RMSE": np.sqrt((residual[:, j] ** 2).mean()),
                "bias": residual[:, j].mean(),
                "median_abs_error": np.median(abs_error[:, j]),
            })

    return pd.DataFrame(rows)

In [ ]:


snr_edges = np.linspace(10, 25, 7)

metrics_snr_val = metrics_by_quantity_bins(
    quantity=snr_val,
    y_true=y_val_phys,
    y_pred=pred_val_phys,
    label_names=label_names,
    bin_edges=snr_edges,
    quantity_name="network_snr"
)


snr_by_label = metrics_snr_val.sort_values(by="label", kind="stable")


display(snr_by_label)

## 6. Save predictions and embeddings

In [ ]:
output_path = RESULTS_DIR / f"{checkpoint_id}_predictions_embeddings.npz"

np.savez(
    output_path,

    pred_train=pred_train,
    pred_val=pred_val,
    pred_cal=pred_cal,
    pred_test=pred_test,

    y_train=y_train,
    y_val=y_val,
    y_cal=y_cal,
    y_test=y_test,

    pred_train_phys=pred_train_phys,
    pred_val_phys=pred_val_phys,
    pred_cal_phys=pred_cal_phys,
    pred_test_phys=pred_test_phys,

    y_train_phys=y_train_phys,
    y_val_phys=y_val_phys,
    y_cal_phys=y_cal_phys,
    y_test_phys=y_test_phys,

    emb_train=emb_train,
    emb_val=emb_val,
    emb_cal=emb_cal,
    emb_test=emb_test,

    idx_train=train_idx,
    idx_val=val_idx,
    idx_cal=cal_idx,
    idx_test=test_idx,

    y_mean=y_mean,
    y_std=y_std,

    label_names=np.array(label_names),

    best_epoch=checkpoint["epoch"],
    best_val_loss=checkpoint["best_val_loss"],
    checkpoint_path=str(checkpoint_path),
)

print("Saved:", output_path)